# Linear Regression Pipeline — Expected Salary Prediction

Steps covered:
1. Load raw CSV
2. Clean missing values (numeric → median imputation)
3. Clean inconsistent categorical values (City: trim whitespace, standardize case)
4. Encode City (One-Hot Encoding)
5. Train/test split + Linear Regression
6. Extract regression equation (coefficients + intercept)
7. Evaluate (R², MAE, RMSE)
8. Sample prediction

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RAW_PATH = "linear_regression_preprocessing_dataset.csv"
CLEAN_PATH = "cleaned_dataset.csv"

## 1. Load the raw dataset

In [2]:
df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
df.head()

Raw shape: (100, 8)


,Student_ID,Study_Hours,Attendance_Percent,Assignments_Completed,Sleep_Hours,Experience_Years,City,Expected_Salary_LPA
0,1001,4.4,56.4,9.0,4.8,3.0,Delhi,95.33
1,1002,9.6,83.6,5.0,8.9,1.0,Noida,132.11
2,1003,7.6,69.1,2.0,8.2,4.0,Gurgaon,115.16
3,1004,6.4,77.9,9.0,8.3,0.0,Noida,104.31
4,1005,2.4,95.8,5.0,5.3,7.0,Noida,98.87


In [3]:
print("Missing values per column (before cleaning):")
df.isna().sum()

Missing values per column (before cleaning):


Student_ID               0
Study_Hours              3
Attendance_Percent       3
Assignments_Completed    2
Sleep_Hours              3
Experience_Years         2
City                     3
Expected_Salary_LPA      0
dtype: int64

## 2. Clean inconsistent City values

Trim whitespace, standardize case (`noida` / `GURGAON` / `Delhi ` → `Noida` / `Gurgaon` / `Delhi`), and fill missing City with the mode.

In [4]:
df["City"] = df["City"].astype(str).str.strip()
df["City"] = df["City"].replace({"nan": np.nan, "": np.nan})
df["City"] = df["City"].str.title()

print("Unique City values after standardizing case/whitespace:", df["City"].unique())

city_mode = df["City"].mode()[0]
df["City"] = df["City"].fillna(city_mode)
print("Filled missing City with mode:", city_mode)

Unique City values after standardizing case/whitespace: <StringArray>
['Delhi', 'Noida', 'Gurgaon', nan]
Length: 4, dtype: str
Filled missing City with mode: Delhi


## 3. Clean missing numeric values

Each numeric column's missing values are filled with that column's median.

In [5]:
numeric_cols = ["Study_Hours", "Attendance_Percent", "Assignments_Completed",
                "Sleep_Hours", "Experience_Years"]

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

target_col = "Expected_Salary_LPA"
df = df.dropna(subset=[target_col])

print("Missing values per column (after cleaning):")
df.isna().sum()

Missing values per column (after cleaning):


Student_ID               0
Study_Hours              0
Attendance_Percent       0
Assignments_Completed    0
Sleep_Hours              0
Experience_Years         0
City                     0
Expected_Salary_LPA      0
dtype: int64

In [6]:
df.to_csv(CLEAN_PATH, index=False)
df.head()

,Student_ID,Study_Hours,Attendance_Percent,Assignments_Completed,Sleep_Hours,Experience_Years,City,Expected_Salary_LPA
0,1001,4.4,56.4,9.0,4.8,3.0,Delhi,95.33
1,1002,9.6,83.6,5.0,8.9,1.0,Noida,132.11
2,1003,7.6,69.1,2.0,8.2,4.0,Gurgaon,115.16
3,1004,6.4,77.9,9.0,8.3,0.0,Noida,104.31
4,1005,2.4,95.8,5.0,5.3,7.0,Noida,98.87


## 4. Encode City (One-Hot Encoding)

`drop_first=True` avoids the dummy variable trap (Delhi becomes the baseline category).

In [7]:
df_encoded = pd.get_dummies(df, columns=["City"], drop_first=True)

feature_cols = ["Study_Hours", "Attendance_Percent", "Assignments_Completed",
                "Sleep_Hours", "Experience_Years"] + \
               [c for c in df_encoded.columns if c.startswith("City_")]

X = df_encoded[feature_cols]
y = df_encoded[target_col]

X.head()

,Study_Hours,Attendance_Percent,Assignments_Completed,Sleep_Hours,Experience_Years,City_Gurgaon,City_Noida
0,4.4,56.4,9.0,4.8,3.0,False,False
1,9.6,83.6,5.0,8.9,1.0,False,True
2,7.6,69.1,2.0,8.2,4.0,True,False
3,6.4,77.9,9.0,8.3,0.0,False,True
4,2.4,95.8,5.0,5.3,7.0,False,True


## 5. Train/test split + Linear Regression

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 6. Regression equation

In [9]:
coefs = model.coef_
intercept = model.intercept_

equation_terms = [f"({coef:.4f} * {name})" for coef, name in zip(coefs, feature_cols)]
equation = f"Expected_Salary_LPA = {intercept:.4f} + " + " + ".join(equation_terms)

print(equation)

Expected_Salary_LPA = 20.4744 + (5.7939 * Study_Hours) + (0.3821 * Attendance_Percent) + (1.4930 * Assignments_Completed) + (1.2705 * Sleep_Hours) + (2.4812 * Experience_Years) + (-2.7445 * City_Gurgaon) + (-2.6451 * City_Noida)


## 7. Evaluation metrics (test set)

In [10]:
y_pred_test = model.predict(X_test)
r2 = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"R2 Score : {r2:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"RMSE     : {rmse:.4f}")

R2 Score : 0.9259
MAE      : 4.2249
RMSE     : 5.3656


## 8. Sample prediction

In [11]:
sample = X_test.iloc[[0]]
sample_actual = y_test.iloc[0]
sample_pred = model.predict(sample)[0]

print("Input features:")
print(sample.to_dict(orient="records")[0])
print(f"Actual Expected_Salary_LPA    : {sample_actual:.2f}")
print(f"Predicted Expected_Salary_LPA : {sample_pred:.2f}")

Input features:
{'Study_Hours': 1.6, 'Attendance_Percent': 94.5, 'Assignments_Completed': 3.0, 'Sleep_Hours': 5.1, 'Experience_Years': 2.0, 'City_Gurgaon': True, 'City_Noida': False}
Actual Expected_Salary_LPA    : 75.97
Predicted Expected_Salary_LPA : 79.03


---
# Part 2: Classification

The original target `Expected_Salary_LPA` is continuous, so to frame this as a classification problem we bucket it into three salary bands — **Low / Medium / High** — using tertiles (quantile-based cuts, so each class is balanced). We then train classifiers to predict the band from the same features used in the regression, and report accuracy, precision, recall, F1-score, and a confusion matrix.

## 9. Create the target classes (salary bands)

In [12]:
from sklearn.preprocessing import LabelEncoder

# Bucket Expected_Salary_LPA into 3 classes using quantiles (tertiles)
df_clf = df.copy()
df_clf["Salary_Band"] = pd.qcut(df_clf["Expected_Salary_LPA"], q=3, labels=["Low", "Medium", "High"])

print(df_clf["Salary_Band"].value_counts())
print("\nBand cutoffs:")
print(pd.qcut(df_clf["Expected_Salary_LPA"], q=3).cat.categories)

Salary_Band
Low       34
Medium    33
High      33
Name: count, dtype: int64

Band cutoffs:
IntervalIndex([(63.539, 95.33], (95.33, 111.11], (111.11, 155.3]], dtype='interval[float64, right]')


## 10. Encode features and target for classification

In [13]:
df_clf_encoded = pd.get_dummies(df_clf, columns=["City"], drop_first=True)

clf_feature_cols = ["Study_Hours", "Attendance_Percent", "Assignments_Completed",
                     "Sleep_Hours", "Experience_Years"] + \
                    [c for c in df_clf_encoded.columns if c.startswith("City_")]

X_clf = df_clf_encoded[clf_feature_cols]

le = LabelEncoder()
y_clf = le.fit_transform(df_clf_encoded["Salary_Band"])
print("Classes:", list(le.classes_), "-> encoded as", list(range(len(le.classes_))))

Classes: ['High', 'Low', 'Medium'] -> encoded as [0, 1, 2]


## 11. Train/test split + train classifiers

We compare **Logistic Regression** and **Random Forest Classifier**.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Scale features for Logistic Regression
scaler = StandardScaler()
Xc_train_scaled = scaler.fit_transform(Xc_train)
Xc_test_scaled = scaler.transform(Xc_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(Xc_train_scaled, yc_train)

rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
rf_clf.fit(Xc_train, yc_train)

print("Models trained: Logistic Regression, Random Forest Classifier")

Models trained: Logistic Regression, Random Forest Classifier


## 12. Evaluation: Accuracy, Precision, Recall, F1

In [15]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"--- {name} ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}\n")
    return acc, prec, rec, f1

y_pred_log = log_reg.predict(Xc_test_scaled)
y_pred_rf = rf_clf.predict(Xc_test)

log_metrics = evaluate("Logistic Regression", yc_test, y_pred_log)
rf_metrics = evaluate("Random Forest", yc_test, y_pred_rf)

--- Logistic Regression ---
Accuracy  : 0.9000
Precision : 0.9062
Recall    : 0.9000
F1-score  : 0.8997

--- Random Forest ---
Accuracy  : 0.8500
Precision : 0.8722
Recall    : 0.8500
F1-score  : 0.8463



## 13. Detailed classification report & confusion matrix

In [16]:
print("=== Logistic Regression: Classification Report ===")
print(classification_report(yc_test, y_pred_log, target_names=le.classes_, zero_division=0))

print("=== Logistic Regression: Confusion Matrix ===")
cm_log = confusion_matrix(yc_test, y_pred_log)
print(pd.DataFrame(cm_log, index=[f"Actual_{c}" for c in le.classes_],
                    columns=[f"Pred_{c}" for c in le.classes_]))

=== Logistic Regression: Classification Report ===
              precision    recall  f1-score   support

        High       1.00      0.86      0.92         7
         Low       0.88      1.00      0.93         7
      Medium       0.83      0.83      0.83         6

    accuracy                           0.90        20
   macro avg       0.90      0.90      0.90        20
weighted avg       0.91      0.90      0.90        20

=== Logistic Regression: Confusion Matrix ===
               Pred_High  Pred_Low  Pred_Medium
Actual_High            6         0            1
Actual_Low             0         7            0
Actual_Medium          0         1            5


In [17]:
print("=== Random Forest: Classification Report ===")
print(classification_report(yc_test, y_pred_rf, target_names=le.classes_, zero_division=0))

print("=== Random Forest: Confusion Matrix ===")
cm_rf = confusion_matrix(yc_test, y_pred_rf)
print(pd.DataFrame(cm_rf, index=[f"Actual_{c}" for c in le.classes_],
                    columns=[f"Pred_{c}" for c in le.classes_]))

=== Random Forest: Classification Report ===
              precision    recall  f1-score   support

        High       0.78      1.00      0.88         7
         Low       0.86      0.86      0.86         7
      Medium       1.00      0.67      0.80         6

    accuracy                           0.85        20
   macro avg       0.88      0.84      0.84        20
weighted avg       0.87      0.85      0.85        20

=== Random Forest: Confusion Matrix ===
               Pred_High  Pred_Low  Pred_Medium
Actual_High            7         0            0
Actual_Low             1         6            0
Actual_Medium          1         1            4


## 14. Feature importance (Random Forest)

In [18]:
importances = pd.Series(rf_clf.feature_importances_, index=clf_feature_cols).sort_values(ascending=False)
print(importances)

Study_Hours              0.411829
Attendance_Percent       0.159993
Sleep_Hours              0.133078
Experience_Years         0.128218
Assignments_Completed    0.114767
City_Gurgaon             0.030344
City_Noida               0.021771
dtype: float64


## 15. Sample classification prediction

In [19]:
sample_clf = Xc_test.iloc[[0]]
sample_clf_scaled = scaler.transform(sample_clf)
actual_band = le.inverse_transform([yc_test[0]])[0]
pred_band_log = le.inverse_transform(log_reg.predict(sample_clf_scaled))[0]
pred_band_rf = le.inverse_transform(rf_clf.predict(sample_clf))[0]

print("Input features:")
print(sample_clf.to_dict(orient="records")[0])
print(f"\nActual Salary Band              : {actual_band}")
print(f"Predicted (Logistic Regression) : {pred_band_log}")
print(f"Predicted (Random Forest)       : {pred_band_rf}")

Input features:
{'Study_Hours': 8.0, 'Attendance_Percent': 85.5, 'Assignments_Completed': 3.0, 'Sleep_Hours': 8.3, 'Experience_Years': 6.0, 'City_Gurgaon': False, 'City_Noida': True}

Actual Salary Band              : High
Predicted (Logistic Regression) : High
Predicted (Random Forest)       : High
